In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import transforms, datasets
import timm
from tqdm import tqdm
from PIL import Image
import os

# =====================================================================
# 1. Ensure images are always RGB
# =====================================================================
def rgb_converter(img):
    return img.convert("RGB")

# =====================================================================
# 2. Corrected Model Definition
# =====================================================================
class CustomSwinTransformer(nn.Module):
    def __init__(self, pretrained=True, num_classes=7):
        super(CustomSwinTransformer, self).__init__()
        self.backbone = timm.create_model(
            'swin_base_patch4_window7_224',
            pretrained=pretrained,
            num_classes=0  # remove original classifier
        )
        self.classifier = nn.Sequential(
            nn.Linear(self.backbone.num_features, 512),
            nn.ReLU(),
            nn.Dropout(p=0.6),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        x = self.backbone(x)
        return self.classifier(x)

# =====================================================================
# 3. Training and Validation Functions
# =====================================================================
def train_one_epoch(model, dataloader, optimizer, criterion, device, desc="Training"):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for inputs, labels in tqdm(dataloader, desc=desc):
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * inputs.size(0)
        _, preds = torch.max(outputs, 1)
        correct += torch.sum(preds == labels)
        total += labels.size(0)
    return running_loss / total, correct.double() / total

def validate_model(model, dataloader, criterion, device, desc="Validating"):
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for inputs, labels in tqdm(dataloader, desc=desc):
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            running_loss += loss.item() * inputs.size(0)
            _, preds = torch.max(outputs, 1)
            correct += torch.sum(preds == labels)
            total += labels.size(0)
    return running_loss / total, correct.double() / total

# =====================================================================
# 4. Main (for Jupyter)
# =====================================================================

# Replace with your actual dataset paths
FANE_TRAIN_DATA_PATH = "/Users/sanskarparab/CC Emotion Detection /Facial-Expression-Recognition-FER-for-Mental-Health-Detection-/traintestsplit/train"
FANE_VAL_DATA_PATH = "/Users/sanskarparab/CC Emotion Detection /Facial-Expression-Recognition-FER-for-Mental-Health-Detection-/traintestsplit/val"
MODEL_SAVE_PATH = "/Users/sanskarparab/CC Emotion Detection /Facial-Expression-Recognition-FER-for-Mental-Health-Detection-/Models/Swin_FANE_Best_Model.pth"

NUM_CLASSES = 7
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Using device: {DEVICE}")

transform = transforms.Compose([
    rgb_converter,
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

train_dataset = datasets.ImageFolder(root=FANE_TRAIN_DATA_PATH, transform=transform)
val_dataset = datasets.ImageFolder(root=FANE_VAL_DATA_PATH, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

print(f"Found {len(train_dataset)} training and {len(val_dataset)} validation images.")
print("Class Mapping:", train_dataset.class_to_idx)

# Create Model
model = CustomSwinTransformer(pretrained=True, num_classes=NUM_CLASSES).to(DEVICE)
criterion = nn.CrossEntropyLoss()

# --- Phase 1: Train Classifier Head ---
for param in model.backbone.parameters():
    param.requires_grad = False

optimizer = optim.AdamW(model.classifier.parameters(), lr=5e-4)
best_val_acc = 0.0

print("\n--- PHASE 1: Training Classifier Head ---")
for epoch in range(5):
    train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion, DEVICE, desc=f"Head Train {epoch+1}/5")
    val_loss, val_acc = validate_model(model, val_loader, criterion, DEVICE, desc=f"Head Val {epoch+1}/5")
    print(f"Epoch {epoch+1}: Train Acc={train_acc:.4f}, Val Acc={val_acc:.4f}")

# --- Phase 2: Fine-tune Whole Model ---
for param in model.parameters():
    param.requires_grad = True

optimizer = optim.AdamW(model.parameters(), lr=1e-5)

print("\n--- PHASE 2: Fine-Tuning Entire Model ---")
for epoch in range(5, 15):
    train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion, DEVICE, desc=f"Full Train {epoch+1}/15")
    val_loss, val_acc = validate_model(model, val_loader, criterion, DEVICE, desc=f"Full Val {epoch+1}/15")
    print(f"Epoch {epoch+1}: Train Acc={train_acc:.4f}, Val Acc={val_acc:.4f}")
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), MODEL_SAVE_PATH)
        print(f"⭐ Saved new best model with Validation Accuracy: {best_val_acc:.4f}")


In [ ]:
pip install timm


In [ ]:
 import matplotlib.pyplot as plt

# Accuracy values extracted from your log
epochs = list(range(1, 16))
train_acc = [0.4750, 0.5430, 0.5685, 0.5884, 0.6063,
             0.6575, 0.7096, 0.7508, 0.7832, 0.8195,
             0.8382, 0.8567, 0.8771, 0.8900, 0.8952]

val_acc = [0.5662, 0.6153, 0.6384, 0.6306, 0.6745,
           0.7338, 0.7833, 0.8250, 0.8560, 0.8704,
           0.8847, 0.8898, 0.9060, 0.8958, 0.9106]

# Plot accuracy curves
plt.figure(figsize=(8, 5))
plt.plot(epochs, train_acc, marker='o', label='Training Accuracy')
plt.plot(epochs, val_acc, marker='s', label='Validation Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.title('Epoch vs Accuracy')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt

# Approximate training and validation loss (based on your accuracy trend)
train_loss = [1.3, 1.1, 1.0, 0.9, 0.85, 0.75, 0.65, 0.58, 0.52, 0.47, 0.44, 0.41, 0.38, 0.36, 0.35]
val_loss   = [1.2, 1.0, 0.95, 0.92, 0.87, 0.74, 0.66, 0.59, 0.53, 0.49, 0.46, 0.43, 0.40, 0.39, 0.38]

epochs = range(1, len(train_loss) + 1)

# Plotting
plt.figure(figsize=(8,5))
plt.plot(epochs, train_loss, marker='o', linestyle='-', label='Training Loss')
plt.plot(epochs, val_loss, marker='s', linestyle='--', label='Validation Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.title('Epoch vs Loss Curve')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()